In [1]:
import torch
import os
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler, LabelEncoder
from functions.running import one_run, plot_results, plot_running_time
import random

In [2]:
def seed_everything(seed=0):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic = True

def run(X, y, dataname):
    # for init_type in ['he', 'xavier', 'orthogonal']:
    for init_type in ['he']:
        epochs = 200
        batch_size = 64
        n_layer = 5
        dataname_ = dataname + str(n_layer) + init_type
        nruns = 10
        res = []
        training_time = []
        for i in range(nruns):
            seed_everything(seed = i)
            res_one_run, training_time_one_run = one_run(init_type, X, y, epochs, batch_size, n_layer, dataname=dataname)
            res.append(res_one_run)
            training_time.append(training_time_one_run)
        res = np.array(res)
        training_time = np.array(training_time)
        mean_all_run = np.mean(res, axis = 0)
        training_time_mean_all_run = np.mean(training_time, axis = 0)
        plot_results(dataname_, mean_all_run, output_dir=f"output/img")
        if not os.path.exists(f"output/res"):
            os.makedirs(f"output/res")
        np.save(f"output/res/"+dataname_, res)

        plot_running_time(dataname_+"_time", training_time_mean_all_run, output_dir=f"output/img")
        if not os.path.exists(f"output/time"):
            os.makedirs(f"output/time")
        np.save(f"output/time/"+dataname_, training_time)


# MNIST

In [1]:
from keras.datasets import mnist

(X_train, y_train), (X_test, y_test) = mnist.load_data()

2025-04-16 07:56:49.306056: I tensorflow/tsl/cuda/cudart_stub.cc:28] Could not find cuda drivers on your machine, GPU will not be used.
2025-04-16 07:56:49.380306: I tensorflow/tsl/cuda/cudart_stub.cc:28] Could not find cuda drivers on your machine, GPU will not be used.
2025-04-16 07:56:49.381550: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2025-04-16 07:56:50.406376: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT


In [4]:
for i in range(10):
    print("train")
    print(i, len(y_train[y_train == i]))

train
0 5923
train
1 6742
train
2 5958
train
3 6131
train
4 5842
train
5 5421
train
6 5918
train
7 6265
train
8 5851
train
9 5949


In [5]:
for i in range(10):
    print("test")
    print(i, len(y_test[y_test == i]))

test
0 980
test
1 1135
test
2 1032
test
3 1010
test
4 982
test
5 892
test
6 958
test
7 1028
test
8 974
test
9 1009


In [3]:
X = None
y = None

run(X, y, 'mnist')

2025-04-15 00:21:41.840278: I tensorflow/tsl/cuda/cudart_stub.cc:28] Could not find cuda drivers on your machine, GPU will not be used.
2025-04-15 00:21:41.903025: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2025-04-15 00:21:42.789048: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT


331
Training with PCA-initialized NN...
Number of PCA components: 331
Epoch 1/30, Training Loss: 0.6504, Testing Accuracy: 0.8849, Training Time: 6.5530
Epoch 2/30, Training Loss: 0.4673, Testing Accuracy: 0.9061, Training Time: 12.7943
Epoch 3/30, Training Loss: 0.5250, Testing Accuracy: 0.8763, Training Time: 19.1119
Epoch 4/30, Training Loss: 0.3638, Testing Accuracy: 0.9159, Training Time: 26.3699
Epoch 5/30, Training Loss: 0.3023, Testing Accuracy: 0.9218, Training Time: 32.5462
Epoch 6/30, Training Loss: 0.2861, Testing Accuracy: 0.9163, Training Time: 38.6966
Epoch 7/30, Training Loss: 0.4088, Testing Accuracy: 0.8876, Training Time: 44.9216
Epoch 8/30, Training Loss: 0.4891, Testing Accuracy: 0.9030, Training Time: 51.2128
Epoch 9/30, Training Loss: 0.3460, Testing Accuracy: 0.9210, Training Time: 58.4680
Epoch 10/30, Training Loss: 0.3135, Testing Accuracy: 0.9299, Training Time: 64.6277
Epoch 11/30, Training Loss: 0.3090, Testing Accuracy: 0.9202, Training Time: 73.8877
Epoch

KeyboardInterrupt: 

# Heart

In [ ]:
data = pd.read_table('https://archive.ics.uci.edu/ml/machine-learning-databases/spect/SPECTF.train', header = None,sep=',')
print(data.head())
test = pd.read_table('https://archive.ics.uci.edu/ml/machine-learning-databases/spect/SPECTF.test',
                     header=None, sep = ',')
data = pd.concat([data, test])
data = data.to_numpy()
X,y = data[:,1:], data[:,0]
G = len(np.unique(y))
print(np.shape(X))
for g in range(G):
  print(sum(y==g))
X.shape
X = X.astype('float')

run(X, y, 'heart')

# Ionosphere

In [ ]:
from sklearn.preprocessing import LabelEncoder
data = pd.read_csv('http://archive.ics.uci.edu/ml/machine-learning-databases/ionosphere/ionosphere.data',
                  sep = ",", header = None)
data = pd.DataFrame.to_numpy(data)
X, y = data[:,:34].astype(np.float64), data[:,34]
le2 = LabelEncoder()
y = le2.fit_transform(y)
G = len(np.unique(y))
X = np.delete(X,[0,1], axis = 1)
for g in range(G):
  print(sum(y==g))
X.shape

run(X, y, 'ionosphere')

# Micromass

In [ ]:
label_data = pd.read_csv('data/micromass/mixed_spectra_metadata.csv', sep = ';')
label = label_data[['Mixture_Label']]
label = label.to_numpy()
print(len(np.unique(label)))
le2 = LabelEncoder()
y = le2.fit_transform(label)

df = pd.read_csv('data/micromass/mixed_spectra_matrix.csv', sep = ';', header = None)
print(df.shape)
print(df.head())
X = df.to_numpy()
var_vec = np.array([np.var(X[:,i]) for i in range(X.shape[1])])
id = np.where(var_vec > 1e-5)
X = X[:,id].reshape((len(X),-1))
X.shape

# run(X, y, 'micromass')

# Parkinson

In [ ]:
data = pd.read_csv('/home/pthnhan/downloads/experiments_PCAInit/data/pd_speech_features.csv', sep = ",", header = [0,1])
print(data.shape)
data.head()
data = data.to_numpy()
X, y = data[:,:-1], data[:,-1]
G = len(np.unique(y))
print('input shape:', X.shape)

run(X, y, 'parkinson')